# Firm × country decomposition of export volatility

Re-does the exercise of Kramarz, Martin & Mejean (JIE 2020) at the **exporter × country** level
instead of the exporter × buyer level.

**Step 1. Cross-sectional decomposition**, year by year, unweighted OLS on continuing pairs in the
connected component:
$$g_{fnt}=\delta_{nzt}+\alpha_{ft}+u_{fnt},$$
with $z$ the firm's main sector in $t-1$, assigned to all of its markets, and $\alpha_{ft}$ normalised
to mean zero by sector-year. Firm aggregation with continuing-market weights $w_{fnt-1}$:
$G_{ft}=C_{ft}+\alpha_{ft}+U_{ft}$.

**Step 2. Conditional variances** over a window of $L$ years before $t$, built on one-year forecast
errors and the weights of $t-1$: $\hat V^C$ (common shocks), $\hat V^\alpha$ (cross-product of two
market halves A/B), $\hat V^U$ ($\sum_n w^2\hat\sigma^2$ with the $N/(N-1)$ correction), plus the
fixed-weight variance of $G$ ($\hat V^G$).

**Step 3. Tests**: (0) cross-sectional $\mathrm{corr}(\hat\alpha,C)$ against its mechanical value under
orthogonal shocks on the same network; (1) the forecast errors of $\alpha$ and $U$ on that of
$C^{-f}$ (cross-fitted on two halves of firms); (2) the gradient of $\hat u_{fn}\hat u_{fm}$ in distance; (3) the global
balance $\hat V^G-(\hat V^C+\hat V^\alpha+\hat V^U)$.

**Inputs**
- `x_eu.dta`, the output of `data_source.do`, with columns `siren, buyer, iso2, year, export, main_cpa, month`.
  Monthly flows are already summed to the year there; they are summed here to siren × iso2 × year.
- Optional: a domestic sales file (`siren, year, sales_dom`, e.g. FICUS/FARE turnover minus exports).
  It adds the market `n = FR`.

With `DATA_PATH = None` the notebook runs on a synthetic economy with known variances. That run
validates the code; its numbers mean nothing about the data.

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ---------------- parameters ----------------
DATA_PATH = None          # path to x_eu.dta; None -> synthetic data
DOMESTIC_PATH = None      # optional .dta/.csv with siren, year, sales_dom -> market "FR"
COUNTRIES = ["BE", "DE", "DK", "ES", "FI", "GB", "IE", "IT", "NL", "PT", "SE"]  # paper's 11
YEARS = (1995, 2007)
TRIM = (-0.8, 4.0)        # bounds on the level growth x_t/x_{t-1}-1 (as in shocks_prepdata.do); None = no trim
MIN_CELL_FIRMS = 2        # a (country, sector) cell needs >= this many firms in the year
L = 5                     # window length (years before t)
FORECAST = "ar1"          # "ar1" (pooled, real-time) or "zero"
MIN_AR_YEARS = 2          # years of (x_tau, x_tau-1) needed before the AR(1) is fitted
MIN_WINDOW = 3            # minimum valid tau in a window
COVERAGE_MIN = 0.9        # minimum t-1 weight with an available forecast error in a tau
K_SHRINK = 5.0            # shrinkage: lambda = m / (m + K_SHRINK), m = number of terms
N_SIM = 20                # simulations for the mechanical corr(alpha, C) benchmark (0 = skip)
SEED = 12345
SYNTH_DOMESTIC = False    # synthetic run only: add a domestic market FR
rng = np.random.default_rng(SEED)

## 0. Data

Annual exports by firm × country, the firm's sector by year (main CPA by export value), and
optionally domestic sales. The sector used in year $t$ is the main sector in $t-1$, so it is fixed
before $t$.

In [ ]:
def load_customs(path, countries, years, chunksize=5_000_000):
    """x_eu.dta -> (siren, iso2, year, export) and (siren, year, main_cpa, export)."""
    xs, ss = [], []
    cols = ["siren", "iso2", "year", "export", "main_cpa"]
    for ch in pd.read_stata(path, columns=cols, chunksize=chunksize):
        ch = ch[ch.iso2.isin(countries) & ch.year.between(*years) & (ch.export > 0)]
        xs.append(ch.groupby(["siren", "iso2", "year"], observed=True)["export"].sum())
        ss.append(ch.groupby(["siren", "year", "main_cpa"], observed=True)["export"].sum())
    x = pd.concat(xs).groupby(level=[0, 1, 2]).sum().reset_index()
    sec = pd.concat(ss).groupby(level=[0, 1, 2]).sum().reset_index()
    return x, sec


def load_domestic(path, years):
    d = pd.read_stata(path) if str(path).endswith(".dta") else pd.read_csv(path)
    d = d[d.year.between(*years) & (d.sales_dom > 0)]
    return d.assign(iso2="FR").rename(columns={"sales_dom": "export"})[["siren", "iso2", "year", "export"]]


NORTH = {"BE", "DE", "DK", "FI", "GB", "IE", "NL", "SE"}


def synthetic_economy(F=4000, S=5, countries=COUNTRIES, years=YEARS, domestic=False,
                      sd_delta=0.08, rho_delta=0.3, sd_alpha=0.15, beta_c=0.5,
                      sd_u=0.30, c_region=0.3, p_active=0.95, seed=1):
    """Known DGP: g = delta_nzt + alpha_ft + u_fnt, alpha_ft = beta_c * (mean_n delta innovations) + nu,
    u = sqrt(1-c) e + sqrt(c) r_{f, region(n), t} (cross-market correlation within region)."""
    r = np.random.default_rng(seed)
    cn = list(countries) + (["FR"] if domestic else [])
    T = np.arange(years[0], years[1] + 1)
    C, nT = len(cn), len(T)
    reg = np.array([c in NORTH for c in cn], int)
    z = r.integers(0, S, F)
    size = r.lognormal(0, 1, F)
    nm = np.minimum(1 + r.poisson(1.5 * size), len(countries))
    pop = np.linspace(2, 1, len(countries)); pop /= pop.sum()
    member = np.zeros((F, C), bool)
    for f in range(F):
        member[f, r.choice(len(countries), nm[f], replace=False, p=pop)] = True
    if domestic:
        member[:, -1] = True
    # delta: AR(1) by (country, sector)
    eta = r.normal(0, sd_delta, (C, S, nT))
    delta = np.zeros((C, S, nT)); delta[..., 0] = eta[..., 0]
    for t in range(1, nT):
        delta[..., t] = rho_delta * delta[..., t - 1] + eta[..., t]
    alpha = np.zeros((F, nT)); u = np.zeros((F, C, nT))
    lvl = np.log(size)[:, None] + r.normal(0, 1, (F, C))
    active = member[..., None] & (r.random((F, C, nT)) < p_active)
    X = np.zeros((F, C, nT))
    for t in range(nT):
        ex = (eta[:, z, t].T * member).sum(1) / member.sum(1)        # mean innovation over the firm's markets
        alpha[:, t] = beta_c * ex + r.normal(0, sd_alpha, F)
        rr = r.normal(0, sd_u, (F, 2))
        u[..., t] = np.sqrt(1 - c_region) * r.normal(0, sd_u, (F, C)) + np.sqrt(c_region) * rr[:, reg]
        if t > 0:
            lvl = lvl + delta[:, z, t].T + alpha[:, [t]] + u[..., t]
        X[..., t] = np.where(active[..., t], np.exp(lvl), 0.0)
    f_, c_, t_ = np.nonzero(X)
    x = pd.DataFrame({"siren": f_.astype(str), "iso2": np.array(cn)[c_], "year": T[t_], "export": X[f_, c_, t_]})
    sec = x[x.iso2 != "FR"].groupby(["siren", "year"], as_index=False)["export"].sum()
    sec["main_cpa"] = z[sec.siren.astype(int).values]
    truth = dict(sd_alpha=sd_alpha, sd_u=sd_u, c_region=c_region, beta_c=beta_c,
                 alpha=pd.DataFrame({"siren": np.repeat(np.arange(F).astype(str), nT), "year": np.tile(T, F),
                                     "alpha_true": alpha.ravel(), "z_true": np.repeat(z, nT)}))
    return x, sec[["siren", "year", "main_cpa", "export"]], truth


SYNTH = DATA_PATH is None
if SYNTH:
    x_raw, sec_raw, TRUTH = synthetic_economy(domestic=SYNTH_DOMESTIC)
else:
    x_raw, sec_raw = load_customs(DATA_PATH, COUNTRIES, YEARS)
    if DOMESTIC_PATH is not None:
        x_raw = pd.concat([x_raw, load_domestic(DOMESTIC_PATH, YEARS)], ignore_index=True)

# integer codes
x_raw["f"] = x_raw.siren.astype("category").cat.codes.astype(np.int64)
FIRM_CODE = dict(zip(x_raw.siren, x_raw.f))
CTRY = sorted(x_raw.iso2.unique()); x_raw["n"] = x_raw.iso2.map({c: i for i, c in enumerate(CTRY)})
sec_raw = sec_raw[sec_raw.siren.isin(FIRM_CODE)].assign(f=lambda d: d.siren.map(FIRM_CODE))
main = (sec_raw.sort_values("export").groupby(["f", "year"]).tail(1)
        .assign(year=lambda d: d.year + 1)[["f", "year", "main_cpa"]].rename(columns={"main_cpa": "zlab"}))
ZLAB = sorted(main.zlab.unique()); main["z"] = main.zlab.map({v: i for i, v in enumerate(ZLAB)})
FZ = main[["f", "year", "z"]]                                     # sector used in year t (= main sector in t-1)
X = x_raw[["f", "n", "year", "export"]]
print(f"{'SYNTHETIC' if SYNTH else 'DATA'}: {X.f.nunique():,} firms, {len(CTRY)} markets {CTRY}, "
      f"{len(ZLAB)} sectors, {len(X):,} firm-country-years")

## 1. Cross-sectional decomposition

Continuing pairs (exports > 0 in $t-1$ and $t$), $g=\ln(x_t/x_{t-1})$, trimmed on level growth.
Each year: cells $(n,z)$ with fewer than `MIN_CELL_FIRMS` firms are pruned, then the largest
connected component of the firm–cell graph is kept in each sector. Because $z$ is a firm attribute,
components never mix sectors, and the mean-zero normalisation of $\alpha$ per sector-year is a pure
normalisation: shifting $\alpha_{ft}$ by $c_z$ and $\delta_{nzt}$ by $-c_z$ leaves every fit unchanged.

Two-way FE are solved by alternating projections. The regression is **not** weighted by $w$: the
first-order condition for $\alpha_{ft}$ is $\sum_n\hat u_{fnt}=0$, so $U_{ft}=\sum_n(w_{fn}-1/N_{ft})\hat u_{fn}$.

In [ ]:
lag = X.assign(year=X.year + 1).rename(columns={"export": "x0"})
P = X.merge(lag, on=["f", "n", "year"])
if TRIM is not None:
    lev = P.export / P.x0 - 1
    P = P[(lev > TRIM[0]) & (lev < TRIM[1])]
P = P.assign(g=np.log(P.export / P.x0)).merge(FZ, on=["f", "year"])
print(f"continuing pairs after trim: {len(P):,}")


def twoway_fe(y, fi, ki, tol=1e-11, maxit=50_000):
    nf, nk = fi.max() + 1, ki.max() + 1
    cf, ck = np.bincount(fi, minlength=nf), np.bincount(ki, minlength=nk)
    d = np.bincount(ki, y, nk) / ck
    for it in range(maxit):
        a = np.bincount(fi, y - d[ki], nf) / cf
        dn = np.bincount(ki, y - a[fi], nk) / ck
        done = np.abs(dn - d).max() < tol
        d = dn
        if done:
            break
    a = np.bincount(fi, y - d[ki], nf) / cf                 # alpha FOC exact
    foc = np.abs(np.bincount(ki, y - a[fi] - d[ki], nk) / ck).max()
    return a, d, it, foc


def estimate_year(d):
    while True:                                             # prune thin cells
        nk = d.groupby(["n", "z"]).f.transform("size")
        if (nk >= MIN_CELL_FIRMS).all():
            break
        d = d[nk >= MIN_CELL_FIRMS]
    fi = d.f.astype("category").cat.codes.values.astype(np.int64)
    kk = d.n.values * 10_000 + d.z.values
    ki = pd.Categorical(kk).codes.astype(np.int64)
    nf, nk = fi.max() + 1, ki.max() + 1
    A = sparse.coo_matrix((np.ones(len(d)), (fi, nf + ki)), shape=(nf + nk, nf + nk))
    _, lab = connected_components(A, directed=False)
    comp = lab[fi]
    key = d.z.values.astype(np.int64) * (comp.max() + 1) + comp
    sz = pd.Series(key).value_counts()
    best = sz.groupby(sz.index // (comp.max() + 1)).idxmax().values   # largest component in each sector
    d = d[np.isin(key, best)].copy()
    fi = d.f.astype("category").cat.codes.values.astype(np.int64)
    ki = pd.Categorical(d.n.values * 10_000 + d.z.values).codes.astype(np.int64)
    a, dl, it, foc = twoway_fe(d.g.values, fi, ki)
    # normalisation: mean-zero alpha per sector-year (over firms)
    fz = pd.Series(d.z.values).groupby(fi).first().values
    cz = pd.Series(a).groupby(fz).mean()
    a = a - cz.reindex(fz).values
    kz = pd.Series(d.z.values).groupby(ki).first().values
    dl = dl + cz.reindex(kz).values
    d["delta"], d["alpha"] = dl[ki], a[fi]
    d["u"] = d.g - d.delta - d.alpha
    d["n_k"] = np.bincount(ki)[ki]
    d["N_f"] = np.bincount(fi)[fi]
    return d, dict(year=int(d.year.iloc[0]), obs=len(d), firms=nf, cells=len(set(ki)), iters=it, foc=foc)


parts, diag = [], []
for t, d in P.groupby("year"):
    e, info = estimate_year(d)
    parts.append(e); diag.append(info)
D = pd.concat(parts, ignore_index=True)
display(pd.DataFrame(diag).set_index("year"))
print(f"kept {len(D):,} of {len(P):,} pairs ({D.export.sum() / P.export.sum():.1%} of value); "
      f"max |g - delta - alpha - u| = {np.abs(D.g - D.delta - D.alpha - D.u).max():.1e}")

In [ ]:
# Firm aggregation with continuing-market weights (sum to one on the estimation sample)
D["w"] = D.x0 / D.groupby(["f", "year"]).x0.transform("sum")
agg = lambda v: (D.w * v).groupby([D.f, D.year]).sum()
FT = pd.DataFrame({"G": agg(D.g), "C": agg(D.delta), "U": agg(D.u),
                   "alpha": D.groupby(["f", "year"]).alpha.first(), "N": D.groupby(["f", "year"]).size(),
                   "H": (D.w ** 2).groupby([D.f, D.year]).sum(), "z": D.groupby(["f", "year"]).z.first()}).reset_index()
print(f"identity G = C + alpha + U: max error {np.abs(FT.G - FT.C - FT.alpha - FT.U).max():.1e}")
print(f"single-market firm-years (U = 0 by construction): {(FT.N == 1).mean():.1%}")
display(FT[["G", "C", "alpha", "U"]].describe().T[["mean", "std", "min", "50%", "max"]])

## 2. Conditional variances

One-year forecast errors $\hat e_{x\tau}=\hat x_\tau-\hat{\mathbb E}_{\tau-1}[\hat x_\tau]$: pooled AR(1)
$\hat x_\tau=a+\rho\hat x_{\tau-1}$ fitted **in real time** on years $<\tau$ (`FORECAST="ar1"`), or zero
forecast. Components: $\delta_{nz}$, $\alpha_f$, its halves $\alpha^A_f,\alpha^B_f$, $u_{fn}$, and $U_f$
(for test 1). Halves: each firm–market pair is assigned once and for all to A or B at random;
$\alpha^{A}_{f\tau}=\mathrm{mean}_{n\in A}(g_{fn\tau}-\hat\delta_{nz\tau})$, the unweighted mean that
defines $\hat\alpha$ itself. Under $\mathrm{Cov}(u_{fn},u_{fm})=0$ the A and B noises are independent,
so $\mathbb E[e^{\alpha,A}e^{\alpha,B}]=\mathrm{Var}(\alpha)$, without the $\bar u_f$ that $\hat\alpha$ contains.

In [ ]:
def realtime_ar1(df, key, col, coef_from=None):
    """Forecast error of `col` on panel `df[key + year]`. Coefficients pooled on years < tau
    (from `coef_from` if given, i.e. (df, col) of another series)."""
    cur = df[key + ["year", col]]
    lagd = cur.assign(year=cur.year + 1).rename(columns={col: "lag"})
    m = cur.merge(lagd, on=key + ["year"], how="left")
    if FORECAST == "zero":
        return m.assign(e=m[col])
    src = m if coef_from is None else coef_from
    scol = col if coef_from is None else coef_from.attrs["col"]
    s = src.dropna(subset=["lag", scol])
    e = np.full(len(m), np.nan)
    for tau in sorted(m.year.unique()):
        past = s[s.year < tau]
        if past.year.nunique() < MIN_AR_YEARS:
            continue
        a, rho = np.polyfit(past.lag.values, past[scol].values, 1)[::-1]
        i = (m.year == tau).values
        e[i] = m[col].values[i] - (a + rho * m.lag.values[i])
    out = m.assign(e=e)
    out.attrs["col"] = col
    return out


# component panels
DEL = D.groupby(["n", "z", "year"]).delta.first().reset_index()
ALP = FT[["f", "year", "alpha"]]
pairs = D[["f", "n"]].drop_duplicates()
pairs["half"] = rng.integers(0, 2, len(pairs))
D = D.drop(columns="half", errors="ignore").merge(pairs, on=["f", "n"])
D["ga"] = D.g - D.delta
halves = D.pivot_table(index=["f", "year"], columns="half", values="ga", aggfunc="mean").reset_index()
halves.columns = ["f", "year", "aA", "aB"]

eD = realtime_ar1(DEL, ["n", "z"], "delta")
eA_full = realtime_ar1(ALP, ["f"], "alpha")
eAA = realtime_ar1(halves[["f", "year", "aA"]].dropna(), ["f"], "aA", coef_from=eA_full)
eAB = realtime_ar1(halves[["f", "year", "aB"]].dropna(), ["f"], "aB", coef_from=eA_full)
eU_obs = realtime_ar1(D[["f", "n", "year", "u"]], ["f", "n"], "u")
eUf = realtime_ar1(FT[["f", "year", "U"]], ["f"], "U")
for nm, s in [("delta", eD), ("alpha", eA_full), ("alpha_A", eAA), ("alpha_B", eAB), ("u", eU_obs), ("U", eUf)]:
    print(f"{nm:8s} forecast errors: {s.e.notna().sum():>9,} non-missing, sd {s.e.std():.3f}")

Window estimators at year $t$ over $\tau\in[t-L,t-1]$, with weights $w_{fn,t-1}$ = shares of **all**
markets active in $t-1$ (information available at $t-1$) and sector $z_{ft}$ (main sector of $t-1$).
A $\tau$ enters if markets carrying at least `COVERAGE_MIN` of the weight have a forecast error;
the weights are renormalised on them. Averages are over the available $\tau$ (at least `MIN_WINDOW`).

- $\hat V^C_{ft}=\mathrm{mean}_\tau\big(\sum_n w_{fn,t-1}\hat e^\delta_{nz\tau}\big)^2$
- $\hat V^\alpha_{ft}=\mathrm{mean}_\tau\,\hat e^{\alpha,A}_{f\tau}\hat e^{\alpha,B}_{f\tau}$, shrunk towards its sector × size-quintile × year mean; the group mean itself is also reported.
- $\hat V^U_{ft}=\sum_n w^2_{fn,t-1}\tilde\sigma^2_{fnt}$, $\hat\sigma^2_{fnt}=\mathrm{mean}_\tau\frac{N_{f\tau}}{N_{f\tau}-1}\hat e^{u\,2}_{fn\tau}$ (terms with $N_{f\tau}\ge2$), shrunk towards the $(n,z,t)$ mean; a market without history gets the group mean.
- $\hat V^G_{ft}=\mathrm{mean}_\tau\big(\sum_n w_{fn,t-1}\hat e^g_{fn\tau}\big)^2$ with $\hat e^g=\hat e^\delta+\hat e^\alpha+\hat e^u$ (the cell of $\tau$); and the raw rolling variance of $\sum_n w_{fn,t-1}g_{fn\tau}$.

In [ ]:
W = X.assign(year=X.year + 1)                                       # weights of t-1, indexed by t
W["w"] = W.export / W.groupby(["f", "year"]).export.transform("sum")
W = W.merge(FZ, on=["f", "year"])[["f", "n", "year", "z", "w"]]
W["nmk"] = W.groupby(["f", "year"]).w.transform("size")
tot0 = X.groupby(["f", "year"]).export.sum().rename("x_tot").reset_index().assign(year=lambda d: d.year + 1)
first_err = int(eD.dropna(subset=["e"]).year.min())
EVAL = [t for t in range(first_err + 1, YEARS[1] + 2) if min(L, t - first_err) >= MIN_WINDOW]
W = W[W.year.isin(EVAL)]


def expand(df):
    """(f, n, t, ...) -> (f, n, t, tau) for tau in [t-L, t-1]."""
    return pd.concat([df.assign(tau=df.year - k) for k in range(1, L + 1)], ignore_index=True)


def window_mean(terms, val):
    g = terms.groupby(["f", "year"])
    return pd.DataFrame({val: g.v.mean(), val + "_m": g.v.size()}).reset_index()


def weighted_tau(Wt, err, on, col="e"):
    """mean over tau of (sum_n w e)^2 with coverage rule."""
    m = Wt.merge(err.rename(columns={"year": "tau"})[on + ["tau", col]], on=on + ["tau"], how="left")
    m = m.assign(we=m.w * m[col], wa=m.w * m[col].notna())
    s = m.groupby(["f", "year", "tau"])[["we", "wa"]].sum().reset_index()
    s = s[s.wa >= COVERAGE_MIN]
    return s.assign(v=(s.we / s.wa) ** 2)


Wt = expand(W)
# V^C
VC = window_mean(weighted_tau(Wt, eD[["n", "z", "year", "e"]], ["n", "z"]), "VC")
# V^alpha (cross-product of halves)
ft_tau = Wt[["f", "year", "tau"]].drop_duplicates()
ab = (ft_tau.merge(eAA[["f", "year", "e"]].rename(columns={"year": "tau", "e": "eA"}), on=["f", "tau"])
            .merge(eAB[["f", "year", "e"]].rename(columns={"year": "tau", "e": "eB"}), on=["f", "tau"]).dropna())
Va = window_mean(ab.assign(v=ab.eA * ab.eB), "Va")
# V^U
eu = eU_obs.merge(D[["f", "n", "year", "N_f"]], on=["f", "n", "year"])
eu = eu[(eu.N_f >= 2) & eu.e.notna()].assign(s2=lambda d: d.N_f / (d.N_f - 1) * d.e ** 2)
sig = Wt.merge(eu[["f", "n", "year", "s2"]].rename(columns={"year": "tau"}), on=["f", "n", "tau"])
sig = sig.groupby(["f", "n", "year"]).s2.agg(["mean", "size"]).reset_index()
VU = W.merge(sig, on=["f", "n", "year"], how="left")
grp = VU.groupby(["n", "z", "year"])["mean"].transform("mean")
lam = VU["size"].fillna(0) / (VU["size"].fillna(0) + K_SHRINK)
VU["s2_shr"] = np.where(VU["size"].fillna(0) > 0, lam * VU["mean"].fillna(0) + (1 - lam) * grp, grp)
VU = VU.assign(t=VU.w ** 2 * VU.s2_shr, hist=VU.w * VU["size"].notna(), w2=VU.w ** 2)
VU = VU.groupby(["f", "year"]).agg(VU=("t", "sum"), VU_hist=("hist", "sum"), H0=("w2", "sum"),
                                     VU_nan=("t", lambda s: s.isna().any())).reset_index()
VU.loc[VU.VU_nan, "VU"] = np.nan
# V^G (forecast errors) and raw rolling variance
eg = (D[["f", "n", "year", "z", "N_f"]]
      .merge(eD[["n", "z", "year", "e"]].rename(columns={"e": "ed"}), on=["n", "z", "year"], how="left")
      .merge(eA_full[["f", "year", "e"]].rename(columns={"e": "ea"}), on=["f", "year"], how="left")
      .merge(eU_obs[["f", "n", "year", "e"]].rename(columns={"e": "eu"}), on=["f", "n", "year"], how="left"))
eg["e"] = eg.ed + eg.ea + eg.eu
VG = window_mean(weighted_tau(Wt, eg[["f", "n", "year", "e"]], ["f", "n"]), "VG")
graw = Wt.merge(D[["f", "n", "year", "g"]].rename(columns={"year": "tau"}), on=["f", "n", "tau"], how="left")
graw = graw.assign(wg=graw.w * graw.g, wa=graw.w * graw.g.notna()).groupby(["f", "year", "tau"])[["wg", "wa"]].sum().reset_index()
graw = graw[graw.wa >= COVERAGE_MIN].assign(v=lambda d: d.wg / d.wa)
VGraw = graw.groupby(["f", "year"]).v.agg(VG_raw="var", VG_raw_m="size").reset_index()

V = (W[["f", "year", "z", "nmk"]].drop_duplicates(["f", "year"]).merge(tot0, on=["f", "year"])
     .merge(VC, on=["f", "year"], how="left").merge(Va, on=["f", "year"], how="left")
     .merge(VU, on=["f", "year"], how="left").merge(VG, on=["f", "year"], how="left")
     .merge(VGraw, on=["f", "year"], how="left"))
for c in ["VC", "Va", "VG", "VG_raw"]:
    V.loc[V[c + "_m"] < MIN_WINDOW, c] = np.nan
V["size_q"] = V.groupby("year").x_tot.transform(lambda s: pd.qcut(np.log(s), 5, labels=False, duplicates="drop"))
V["Va_grp"] = V.groupby(["z", "size_q", "year"]).Va.transform("mean")
lam = V.Va_m.fillna(0) / (V.Va_m.fillna(0) + K_SHRINK)
V["Va_shr"] = np.where(V.Va.notna(), lam * V.Va + (1 - lam) * V.Va_grp, V.Va_grp)
V["gap"] = V.VG - V.VC - V.Va_shr - V.VU
VV = V.dropna(subset=["VC", "Va_shr", "VU", "VG"])
print(f"evaluation years {EVAL}; firm-years with all four variances: {len(VV):,} of {len(V):,}")
print(f"raw V^alpha < 0 in {(V.Va < 0).mean():.1%} of firm-years (before shrinkage)")
display(VV[["VG", "VG_raw", "VC", "Va", "Va_shr", "Va_grp", "VU", "gap"]].describe().T[["mean", "50%", "std"]])

## 3. Tests

### 3.0 Cross-sectional $\mathrm{corr}(\hat\alpha_{ft},C_{ft})$ against its mechanical value
This moment is free in OLS but biased by low connectivity (limited-mobility bias). The benchmark
re-estimates the model on the **same network** year by year with independent normal $\delta,\alpha,u$
at the estimated variances: the simulated correlation is what the estimator produces when the true
one is zero.

In [ ]:
def corr_alpha_C(d):
    w = d.x0 / d.groupby("f").x0.transform("sum")
    c = (w * d.delta).groupby(d.f).sum()
    return np.corrcoef(d.groupby("f").alpha.first(), c)[0, 1]


rows = []
for t, d in D.groupby("year"):
    fi = d.f.astype("category").cat.codes.values.astype(np.int64)
    ki = pd.Categorical(d.n.values * 10_000 + d.z.values).codes.astype(np.int64)
    multi = d.N_f.values >= 2
    sd_d, sd_a = d.groupby(["n", "z"]).delta.first().std(), d[multi].groupby("f").alpha.first().std()
    sd_u = np.sqrt((d.u[multi] ** 2 * d.N_f[multi] / (d.N_f[multi] - 1)).mean())
    sims = []
    for s in range(N_SIM):
        y = (rng.normal(0, sd_d, ki.max() + 1)[ki] + rng.normal(0, sd_a, fi.max() + 1)[fi]
             + rng.normal(0, sd_u, len(d)))
        a, dl, _, _ = twoway_fe(y, fi, ki, tol=1e-9)
        sims.append(corr_alpha_C(d.assign(alpha=a[fi], delta=dl[ki])))
    rows.append(dict(year=t, corr_data=corr_alpha_C(d), corr_mech=np.mean(sims) if sims else np.nan,
                     sd_mech=np.std(sims) if sims else np.nan))
T0 = pd.DataFrame(rows).set_index("year")
T0["corr_net"] = T0.corr_data - T0.corr_mech
display(T0)

### 3.1 Term (3): forecast errors of $\alpha$ and $U$ on that of $C^{-f}$ (cross-fitted)
A leave-one-out $\hat\delta^{-f}$ is not enough: $\hat\alpha_f=\mathrm{mean}_n(g_{fn}-\hat\delta_n)$ contains
$-\mathrm{mean}_n(\hat\delta_n-\delta_n)$, and that estimation error comes from the other firms of the cell,
which also drive $\hat\delta^{-f}$. Under the null this creates a negative slope; in the synthetic
check it gave $t\approx-2.6$. Cross-fitting removes it. Firms are split once and for all into two
halves $h\in\{0,1\}$ and the model is re-estimated on each half separately. For a firm in half $h$,
the dependent variables $\hat e^\alpha,\hat e^U$ come from half $h$, and the regressor
$\hat e^{C,-f}_{ft}=\sum_n w_{fn,t-1}\big(\hat\delta^{1-h}_{nzt}-\hat{\mathbb E}_{t-1}[\delta^{1-h}_{nz}]\big)$ from the other
half. The two halves have independent noises, so under the null the slope is zero. Each half has
its own normalisation constant per sector-year; the sector-year FE absorb it.

Firm and sector-year FE (alternating projections), unweighted. Standard errors: clustered by
sector-year and by firm, plus an AKM-style shift-share variance
$\hat V=\sum_k\tilde\varepsilon_k^2\big(\sum_i s_{ik}\hat\eta_i\big)^2/\big(\sum_i\tilde X_i^2\big)^2$, where the shocks are
$k=(n,z,t)$, $\varepsilon_k$ is the full-sample $\hat e^\delta_k$ demeaned within sector-year, and the shares
are $s_{ik}=w_{fn,t-1}$. This approximates Adão, Kolesár & Morales (2019): the shock residualisation
ignores the firm FE.

In [ ]:
def demean2(v, g1, g2, tol=1e-12, maxit=10_000):
    v = v.astype(float).copy()
    c1, c2 = np.bincount(g1), np.bincount(g2)
    for _ in range(maxit):
        m1 = np.bincount(g1, v) / c1; v -= m1[g1]
        m2 = np.bincount(g2, v) / c2; v -= m2[g2]
        if max(np.abs(m1).max(), np.abs(m2).max()) < tol:
            break
    return v


def cluster_se(x, r, cl, sxx):
    s = pd.Series(x * r).groupby(cl).sum().values
    G = len(s)
    return np.sqrt(G / (G - 1) * (s ** 2).sum()) / sxx


# re-estimate on each half of firms
half_of = rng.integers(0, 2, D.f.max() + 1)
XF = []
for t, d in D.groupby("year"):
    d = d[["f", "n", "z", "year", "g", "x0"]]
    for h in (0, 1):
        e, _ = estimate_year(d[half_of[d.f.values] == h])
        XF.append(e.assign(h=h)[["f", "n", "z", "year", "h", "delta", "alpha", "u", "x0"]])
XF = pd.concat(XF, ignore_index=True)
XF["w"] = XF.x0 / XF.groupby(["f", "year"]).x0.transform("sum")
eDh = realtime_ar1(XF.groupby(["h", "n", "z", "year"]).delta.first().reset_index(), ["h", "n", "z"], "delta", coef_from=eD)
eAh = realtime_ar1(XF.groupby(["f", "year"]).alpha.first().reset_index(), ["f"], "alpha", coef_from=eA_full)
eUh = realtime_ar1((XF.w * XF.u).groupby([XF.f, XF.year]).sum().rename("U").reset_index(), ["f"], "U", coef_from=eUf)
# regressor: own markets and weights, other half's delta errors
ec = XF[["f", "n", "z", "year", "h", "w"]].assign(h=lambda d: 1 - d.h)
ec = ec.merge(eDh[["h", "n", "z", "year", "e"]], on=["h", "n", "z", "year"], how="left")
ec = ec.assign(we=ec.w * ec.e, wa=ec.w * ec.e.notna()).groupby(["f", "year"])[["we", "wa"]].sum().reset_index()
ec = ec[ec.wa >= COVERAGE_MIN].assign(eC_x=lambda d: d.we / d.wa)[["f", "year", "eC_x"]]
R1 = (ec.merge(eAh[["f", "year", "e"]].rename(columns={"e": "e_alpha"}), on=["f", "year"])
        .merge(eUh[["f", "year", "e"]].rename(columns={"e": "e_U"}), on=["f", "year"])
        .merge(FT[["f", "year", "z"]], on=["f", "year"]).dropna())
R1 = R1[R1.groupby("f").f.transform("size") > 1].reset_index(drop=True)
g1 = R1.f.astype("category").cat.codes.values.astype(np.int64)
g2 = pd.Categorical(R1.z.astype(str) + "_" + R1.year.astype(str)).codes.astype(np.int64)
xt = demean2(R1.eC_x.values, g1, g2)
sxx = (xt ** 2).sum()
# AKM ingredients: shares on continuing markets, full-sample shocks demeaned within (z, t)
sh = XF[["f", "n", "z", "year", "w"]].merge(R1[["f", "year"]].assign(i=np.arange(len(R1))), on=["f", "year"])
sh = sh.merge(eD[["n", "z", "year", "e"]], on=["n", "z", "year"], how="left").dropna(subset=["e"])
sh["eps"] = sh.e - sh.groupby(["z", "year"]).e.transform("mean")
out = []
for dep in ["e_alpha", "e_U"]:
    yt = demean2(R1[dep].values, g1, g2)
    b = (xt * yt).sum() / sxx
    r = yt - b * xt
    Rk = (sh.w * r[sh.i.values]).groupby([sh.n, sh.z, sh.year]).sum()
    ek = sh.groupby(["n", "z", "year"]).eps.first().reindex(Rk.index)
    out.append(dict(dep=dep, beta=b, se_sector_year=cluster_se(xt, r, g2, sxx), se_firm=cluster_se(xt, r, g1, sxx),
                    se_akm=np.sqrt((ek ** 2 * Rk ** 2).sum()) / sxx, nobs=len(R1)))
T1 = pd.DataFrame(out).set_index("dep")
T1["t_akm"] = T1.beta / T1.se_akm
display(T1)

### 3.2 Cross-market correlation of $\hat u$: gradient in distance
For each firm-year with $N\ge3$ markets and every pair $n<m$: $p=\hat u_{fn}\hat u_{fm}/s^2_{ft}$,
$s^2_{ft}=\mathrm{mean}_n\hat u^2_{fn}$ (scale-free). Regressed on log distance between capitals and
contiguity, with firm-year FE and country-membership dummies $D_n+D_m$. The level of
$\mathrm{Cov}(u_{fn},u_{fm})$ is not identified (absorbed by $\alpha$); under
$\mathrm{Cov}(u_{fn},u_{fm})=0$, $\mathbb E[\hat u_n\hat u_m]$ depends on $(\sigma^2_n+\sigma^2_m)/N$, which the
country dummies absorb. A negative distance slope rejects the null. SEs clustered by firm.

In [ ]:
CAP = {"AT": (48.21, 16.37), "BE": (50.85, 4.35), "DE": (52.52, 13.40), "DK": (55.68, 12.57), "ES": (40.42, -3.70),
       "FI": (60.17, 24.94), "FR": (48.86, 2.35), "GB": (51.51, -0.13), "GR": (37.98, 23.73), "IE": (53.35, -6.26),
       "IT": (41.90, 12.50), "LU": (49.61, 6.13), "NL": (52.37, 4.90), "PT": (38.72, -9.14), "SE": (59.33, 18.07)}
CONTIG = {frozenset(p) for p in [("BE", "DE"), ("BE", "NL"), ("BE", "LU"), ("BE", "FR"), ("DE", "NL"), ("DE", "DK"),
                                 ("DE", "AT"), ("DE", "LU"), ("DE", "FR"), ("ES", "PT"), ("ES", "FR"), ("IT", "FR"),
                                 ("IT", "AT"), ("LU", "FR"), ("GB", "IE"), ("FI", "SE")]}


def gc_km(a, b):
    (la1, lo1), (la2, lo2) = np.radians(CAP[a]), np.radians(CAP[b])
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(h))


d3 = D.loc[D.N_f >= 3, ["f", "year", "n", "u"]]
d3 = d3.assign(s2=d3.groupby(["f", "year"]).u.transform(lambda s: (s ** 2).mean()))
d3 = d3[d3.s2 > 0]
pp = d3.merge(d3[["f", "year", "n", "u"]], on=["f", "year"], suffixes=("", "_m"))
pp = pp[pp.n < pp.n_m]
pp["p"] = pp.u * pp.u_m / pp.s2
dist = np.array([[np.log(gc_km(a, b)) if a in CAP and b in CAP and a != b else np.nan for b in CTRY] for a in CTRY])
cont = np.array([[float(frozenset((a, b)) in CONTIG) for b in CTRY] for a in CTRY])
pp["ldist"], pp["contig"] = dist[pp.n, pp.n_m], cont[pp.n, pp.n_m]
pp = pp.dropna(subset=["ldist"])
gfy = pd.Categorical(pp.f.astype(str) + "_" + pp.year.astype(str)).codes.astype(np.int64)
cf = np.bincount(gfy)
dm = lambda v: v - (np.bincount(gfy, v) / cf)[gfy]
Xc = [pp.ldist.values, pp.contig.values] + [((pp.n == j) | (pp.n_m == j)).astype(float).values for j in range(1, len(CTRY))]
Xm = np.column_stack([dm(v) for v in Xc])
ym = dm(pp.p.values)
bhat, *_ = np.linalg.lstsq(Xm, ym, rcond=None)
res = ym - Xm @ bhat
XtX_inv = np.linalg.pinv(Xm.T @ Xm)
sc = pd.DataFrame(Xm * res[:, None]).groupby(pp.f.values).sum().values
Gc = len(sc)
Vb = XtX_inv @ (sc.T @ sc) @ XtX_inv * Gc / (Gc - 1)
T2 = pd.DataFrame({"coef": bhat[:2], "se_firm": np.sqrt(np.diag(Vb))[:2]}, index=["log distance", "contiguity"])
T2["t"] = T2.coef / T2.se_firm
print(f"{len(pp):,} market pairs, {Gc:,} firms")
display(T2)

### 3.3 Global balance
$\hat V^G-(\hat V^C+\hat V^\alpha+\hat V^U)$ estimates $2[\mathrm{Cov}(C,\alpha)+\mathrm{Cov}(C,U)+\mathrm{Cov}(\alpha,U)]$ plus the weighted
cross-market covariance of $u$ and estimation noise in $\hat\delta$. Means across firm-years
(medians of a difference are not differences of medians), overall and by size quintile in $t-1$.

In [ ]:
cols = ["VG", "VC", "Va_shr", "VU", "gap"]
T3 = VV.groupby("size_q")[cols].mean()
T3.loc["all"] = VV[cols].mean()
T3["gap/VG"] = T3.gap / T3.VG
T3["N"] = pd.concat([VV.groupby("size_q").size(), pd.Series({"all": len(VV)})])
display(T3)

### Synthetic run only: compare with the known DGP
Oracle $V^\alpha$ is the mean of the true $\alpha_{ft}^2$ after the sector-year normalisation; oracle
$V^U$ is $\sigma_u^2\sum_n w^2$. The DGP has $\mathrm{Cov}(\alpha,C)>0$ through `beta_c` and a positive
within-region correlation of $u$, so tests 3.1 (on $e^\alpha$) and 3.2 should reject, and the gap of 3.3
should be positive.

In [ ]:
if SYNTH:
    tr = TRUTH["alpha"].assign(f=lambda d: d.siren.map(FIRM_CODE)).dropna(subset=["f"])
    tr["a_n"] = tr.alpha_true - tr.groupby(["z_true", "year"]).alpha_true.transform("mean")
    orc = expand(V[["f", "year"]]).merge(tr[["f", "year", "a_n"]].rename(columns={"year": "tau"}), on=["f", "tau"])
    orc = orc.groupby(["f", "year"]).a_n.apply(lambda s: (s ** 2).mean()).rename("Va_oracle").reset_index()
    chk = VV.merge(orc, on=["f", "year"])
    print(f"V^alpha: estimated mean {chk.Va_shr.mean():.4f} (raw {chk.Va.mean():.4f}) vs oracle {chk.Va_oracle.mean():.4f}")
    print(f"V^U:     estimated mean {chk.VU.mean():.4f} vs oracle sigma_u^2 * H = {(TRUTH['sd_u'] ** 2 * chk.H0).mean():.4f}")
    print(f"slope of e_alpha on e_C(-f): {T1.loc['e_alpha', 'beta']:.3f} (DGP: alpha loads {TRUTH['beta_c']} on the "
          f"unweighted mean delta innovation)")

In [ ]:
OUT = None   # e.g. "results_firm_country" to save
if OUT:
    import os
    os.makedirs(OUT, exist_ok=True)
    D.to_parquet(f"{OUT}/pairs_decomposition.parquet")
    FT.to_parquet(f"{OUT}/firm_year_decomposition.parquet")
    V.to_parquet(f"{OUT}/conditional_variances.parquet")
    for nm, tb in [("diag", pd.DataFrame(diag)), ("test0", T0), ("test1", T1), ("test2", T2), ("test3", T3)]:
        tb.to_csv(f"{OUT}/{nm}.csv")